In [5]:
import pandas as pd
import numpy as np
import os, requests
from concurrent.futures import ThreadPoolExecutor
from PIL import Image
from io import BytesIO

In [6]:
# paths
BASE_DIR  = os.getcwd()
IMAGE_DIR = os.path.join(BASE_DIR, 'images')
os.makedirs(IMAGE_DIR, exist_ok=True)

In [7]:
# load
df = pd.read_csv(os.path.join(BASE_DIR, 'housing.csv'))
print(f"After load:            {len(df)}")

After load:            8983


In [8]:
# clean csv
df = df.drop(columns=['Unnamed: 2', 'Unnamed: 40'], errors='ignore')

df['price'] = pd.to_numeric(df['price'].astype(str).str.replace(r'[\$,\s]', '', regex=True), errors='coerce')

for col in ['sqft', 'sqft_lot']:
    if df[col].dtype == object:
        df[col] = df[col].str.replace(',', '').str.extract(r'(\d+\.?\d*)').astype(float)

if df['price_per_sqft'].dtype == object:
    df['price_per_sqft'] = df['price_per_sqft'].str.replace(r'[\$,/sqft\s]', '', regex=True).astype(float)

for col in ['walk_score', 'bike_score']:
    if col in df.columns and df[col].dtype == object:
        df[col] = df[col].str.extract(r'(\d+)').astype(float)

if df['estimated_monthly_payment'].dtype == object:
    df['estimated_monthly_payment'] = df['estimated_monthly_payment'].str.extract(r'(\d+\.?\d*)').astype(float)

df['property_type'] = df['property_type'].str.lower().str.strip()
df = df.drop(columns=['transit_score'], errors='ignore')

for col in ['baths', 'year_built', 'walk_score', 'bike_score', 'parking_total_spaces']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

df = df.dropna(subset=['price', 'image_url', 'beds', 'sqft'])
print(f"After dropna:          {len(df)}")

q_low  = df['price'].quantile(0.01)
q_high = df['price'].quantile(0.99)
df = df[(df['price'] >= q_low) & (df['price'] <= q_high)]
print(f"After outlier removal: {len(df)}")

After dropna:          8575
After outlier removal: 8406


In [9]:
# download images
def download_image(row):
    img_path = os.path.join(IMAGE_DIR, f"{row.name}.jpg")
    if os.path.exists(img_path):
        return img_path
    try:
        r = requests.get(row['image_url'], timeout=10, headers={'User-Agent': 'Mozilla/5.0'})
        img = Image.open(BytesIO(r.content)).convert('RGB')
        img = img.resize((224, 224))
        img.save(img_path)
        return img_path
    except:
        return None

print("Downloading images...")
with ThreadPoolExecutor(max_workers=8) as ex:
    df['img_path'] = list(ex.map(download_image, [row for _, row in df.iterrows()]))

print(f"Downloaded: {df['img_path'].notna().sum()} / {len(df)}")

Downloaded: 8398 / 8406


In [10]:
# save cleaned csv
df.to_csv(os.path.join(BASE_DIR, 'listings_clean.csv'), index=False)
print("Saved listings_clean.csv")

Saved listings_clean.csv


In [11]:
df_clean = pd.read_csv(os.path.join(BASE_DIR, 'listings_clean.csv'))
df_clean.head()

,url,image_url,beds,baths,sqft,sqft_lot,address,estimated_monthly_payment,property_type,price_per_sqft,...,fire_risk,wind_risk,air_risk,heat_risk,nearby_cities,property_history,parking_uncovered_spaces,parking_garage_spaces,price,img_path
0,https://www.zillow.com/homedetails/197-Lowell-...,https://photos.zillowstatic.com/fp/90454dcdc33...,3.0,2.0,"1,415","5,001 sqft","197 Lowell St, Arlington, MA 02474","$6,225/mo",single family,$706/sqft,...,Minimal (1/10),Major (6/10),Moderate (3/10),Major (5/10),Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/15/2025', 'event': 'Listed for sal...",NaN,NaN,999000,/Users/user/Documents/undergrad-cs/computer-vi...
1,https://www.zillow.com/homedetails/225-Mystic-...,https://photos.zillowstatic.com/fp/e92a0b0b33e...,2.0,1.0,"1,064","4,818 sqft","225 Mystic St, Arlington, MA 02474","$4,978/mo",single family,$751/sqft,...,Minimal (1/10),Major (6/10),Moderate (3/10),Major (5/10),Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '6/25/2025', 'event': 'Listed for sal...",yes,NaN,799000,/Users/user/Documents/undergrad-cs/computer-vi...
2,https://www.zillow.com/homedetails/43-Longmead...,https://photos.zillowstatic.com/fp/740e93bfcff...,2.0,1.0,"1,628","7,954 sqft","43 Longmeadow Rd, Arlington, MA 02474","$5,603/mo",single family,$552/sqft,...,Minimal (1/10),Major (6/10),Moderate (3/10),Major (5/10),Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/22/2025', 'event': 'Listed for sal...",NaN,NaN,899000,/Users/user/Documents/undergrad-cs/computer-vi...
3,https://www.zillow.com/homedetails/11-Pine-Ct-...,https://photos.zillowstatic.com/fp/bb84e595f96...,3.0,2.0,"1,824","6,037 sqft","11 Pine Ct, Arlington, MA 02476","$6,219/mo",single family,$547/sqft,...,Minimal (1/10),Major (6/10),Moderate (3/10),Major (6/10),Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '7/22/2025', 'event': 'Listed for sal...",NaN,NaN,998000,/Users/user/Documents/undergrad-cs/computer-vi...
4,https://www.zillow.com/homedetails/17-Norcross...,https://photos.zillowstatic.com/fp/ac474eb496d...,2.0,2.0,"1,221",NaN,"17 Norcross St FLOOR 3, Arlington, MA 02474","$4,339/mo",condo,$547/sqft,...,Minimal (1/10),Major (6/10),Moderate (3/10),Major (6/10),Arlington; Cambridge; Everett; Framingham; Lowell,"{'date': '6/21/2025', 'event': 'Price change',...",yes,NaN,668000,/Users/user/Documents/undergrad-cs/computer-vi...
